# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# View metadata
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Keywords: {dataset.metadata.keywords}")
print(f"Date published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

A Croissant dataset organizes data into record sets (tables or main data sections), each with fields (columns) and possibly sub-structure. We list the available record sets and their fields below:

In [ ]:
# List all available record sets with their '@id' and name
record_sets = list(dataset.metadata.recordSet or [])
if len(record_sets) == 0:
    # Try getting from dataset.metadata.record_sets (in case of alternate attribute)
    try:
        record_sets = list(dataset.metadata.record_sets)
    except Exception:
        pass

if not record_sets:
    print("No record sets found in the dataset metadata.\nCheck the schema for the record sets location.")
else:
    for rs in record_sets:
        print(f"RecordSet: @id={rs['@id']} - name: {rs.get('name', '')}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        field_ids = [f["@id"] for f in fields]
        print(f"  Fields (@id): {field_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. If you're not sure which record set has the main data, look for one related to protein or peptide entries.

In [ ]:
# Set up record set(s) for data extraction
# (Replace these with concrete @id's from your previous code output)
# For this demo, we'll show example placeholders.

# Example: Insert your record set @id here!
record_sets_ids = [
    # e.g., "cr:ProteinTable" # <-- Replace with real record set @id
]
if not record_sets_ids:
    print("Please set your record_sets_ids to at least one record set @id, as shown in the previous section.")
else:
    dataframes = {}
    for record_set in record_sets_ids:
        records = list(dataset.records(record_set=record_set))
        dataframes[record_set] = pd.DataFrame(records)

    print("Extracted columns for each record set:")
    for rid, df in dataframes.items():
        print(f"@id={rid}: {df.columns.tolist()}")
    # Show first rows for the primary record set
    main_id = record_sets_ids[0]
    display(dataframes[main_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

You must reference columns by their field `@id` (as seen above).

In [ ]:
# Example EDA for a numeric field
# Fill in your numeric field's @id (e.g., 'cr:mw' for molecular weight)

main_record_set_id = None
numeric_field_id = None
group_field_id = None

# Example (Replace below with real values from your dataset):
# main_record_set_id = "cr:ProteinTable"
# numeric_field_id = "cr:mw" # e.g., molecular weight
# group_field_id = "cr:sample_id" # if available, for grouping

if not main_record_set_id or not numeric_field_id:
    print("Please set 'main_record_set_id' and 'numeric_field_id' to your target record set and field @id, as shown in the previous sections.")
else:
    df = dataframes[main_record_set_id]
    # Convert the column to numeric (if it's not already)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping
    if group_field_id and (group_field_id in filtered_df.columns):
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the selected numeric field. Adjust `main_record_set_id` and `numeric_field_id` as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not main_record_set_id or not numeric_field_id:
    print("Set main_record_set_id and numeric_field_id above to plot their distribution.")
else:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains mass spectrometry results for extracellular vesicles from human mast cells, with extensive metadata.
- Using the `mlcroissant` library, data can be loaded, fields referenced programmatically by `@id`, and exploratory data analysis applied.
- Remember to always use `@id` fields for referencing entities for full schema compatibility and reproducibility.

You can further extend this notebook by adding machine learning analysis, domain-specific visualizations, or advanced feature engineering tailored to mass spectrometry and proteomics data.